In [22]:
!pip install nba_api

In [23]:
# Cell 1 is for the import statements and the setup for the project
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.static import teams
import time

nba_teams = teams.get_teams()
print(f"Total teams: {len(nba_teams)}")
print(nba_teams[:5])

Total teams: 30
[{'id': 1610612737, 'full_name': 'Atlanta Hawks', 'abbreviation': 'ATL', 'nickname': 'Hawks', 'city': 'Atlanta', 'state': 'Georgia', 'year_founded': 1949}, {'id': 1610612738, 'full_name': 'Boston Celtics', 'abbreviation': 'BOS', 'nickname': 'Celtics', 'city': 'Boston', 'state': 'Massachusetts', 'year_founded': 1946}, {'id': 1610612739, 'full_name': 'Cleveland Cavaliers', 'abbreviation': 'CLE', 'nickname': 'Cavaliers', 'city': 'Cleveland', 'state': 'Ohio', 'year_founded': 1970}, {'id': 1610612740, 'full_name': 'New Orleans Pelicans', 'abbreviation': 'NOP', 'nickname': 'Pelicans', 'city': 'New Orleans', 'state': 'Louisiana', 'year_founded': 2002}, {'id': 1610612741, 'full_name': 'Chicago Bulls', 'abbreviation': 'CHI', 'nickname': 'Bulls', 'city': 'Chicago', 'state': 'Illinois', 'year_founded': 1966}]


In [24]:
# Get the results across the past 5 seasons to compare to the current season at the end for "accuracy" in a way
seasons = ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
all_games= []

for season in seasons:
  print(f"Pulling seasons: {season}")
  gamefinder = leaguegamefinder.LeagueGameFinder(season_nullable=season)
  games = gamefinder.get_data_frames()[0]
  all_games.append(games)
  time.sleep(2)

df = pd.concat(all_games, ignore_index=True)
print(f"\nTotal rows: {len(df)}")
print(df.columns.tolist())
print(df.head())

# Following this output, each row is one team's performance in one game, so every game has two rows (one per team). The columns display the stats of a team points,
# FG%, assists, steals, rebounds, blocks, turnovers, plus/minus, and win/loss. This is the skeleton of a dataset we will use to build features and train models.

Pulling seasons: 2019-20
Pulling seasons: 2020-21
Pulling seasons: 2021-22
Pulling seasons: 2022-23
Pulling seasons: 2023-24
Pulling seasons: 2024-25

Total rows: 16133
['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS']
  SEASON_ID     TEAM_ID TEAM_ABBREVIATION           TEAM_NAME     GAME_ID  \
0     42019  1610612747               LAL  Los Angeles Lakers  0041900406   
1     42019  1610612748               MIA          Miami Heat  0041900406   
2     42019  1610612748               MIA          Miami Heat  0041900405   
3     42019  1610612747               LAL  Los Angeles Lakers  0041900405   
4     42019  1610612748               MIA          Miami Heat  0041900404   

    GAME_DATE      MATCHUP WL  MIN  PTS  ...  FT_PCT  OREB  DREB  REB  AST  \
0  2020-10-11    LAL @ MIA  W 

In [25]:
# figuring out the dataframe shape, what datatypes are each column, and figuring out if there are missing values I will need to deal with

print(df.shape)
print(df.dtypes)
print(df.isnull().sum)

(16133, 28)
SEASON_ID             object
TEAM_ID                int64
TEAM_ABBREVIATION     object
TEAM_NAME             object
GAME_ID               object
GAME_DATE             object
MATCHUP               object
WL                    object
MIN                    int64
PTS                    int64
FGM                    int64
FGA                    int64
FG_PCT               float64
FG3M                   int64
FG3A                   int64
FG3_PCT              float64
FTM                    int64
FTA                    int64
FT_PCT               float64
OREB                   int64
DREB                   int64
REB                    int64
AST                    int64
STL                    int64
BLK                    int64
TOV                    int64
PF                     int64
PLUS_MINUS           float64
dtype: object
<bound method DataFrame.sum of        SEASON_ID  TEAM_ID  TEAM_ABBREVIATION  TEAM_NAME  GAME_ID  GAME_DATE  \
0          False    False              False      Fa